# Building design matrices and residuals for the fsLR data

A walkthrough of `design_matrix.py`, which builds a first-level design matrix
for one run of the network-grant discovery data in fsLR (den-91k) space.

Four parts:

1. **Setup** — paths and identifiers
2. **Building a design matrix** — and what every output means
3. **Choosing confounds** — motion, tissue, DCT, global signal
4. **Fitting and residuals**

### How a task is defined

Everything about a task lives in one YAML file. Each regressor has exactly
three fields:

```yaml
congruent:
  amplitude: 1                 # a number, or an events column for parametric modulation
  duration: duration           # an events column, or a number for a fixed epoch
  subset: >-                   # a pandas query picking rows from the events file
    trial_id == 'test_trial' and trial_type == 'congruent'
    and key_press == correct_response and response_time >= 0.2
```

Every name in a YAML is a column that actually exists in the events.tsv, so
you can open the two side by side and see exactly what the model is. There are
no columns computed behind your back, and no task names anywhere in the Python
— **adding a task means writing one YAML and changing no code.** A task with no
YAML raises an error.

### Important things to know before you start

**The events and imaging files on OAK are untrimmed (as of 9/18/2026).** The first 7 volumes of
each run are warmup (non-steady-state) volumes and the events onsets are raw.
This module removes those volumes, shifts the onsets by −7 × TR = −10.43 s, and
drops the matching 7 rows from the confounds. That is what
`data_is_trimmed=False` means, and it has no default — you must say which state
your data are in, because **nothing in the files records it.** A wrong value
gives a silent 10.43 s misalignment, not an error.

How can you check if the data were trimmed?  In the fmriprep derivatives, find `sub-s03_ses-01_task-spatialTS_run-1_space-T1w_desc-preproc_bold.nii.gz`
and check the number of volumes.  If this image has 344 volumes, it has not been trimmed.  Also check the `sub-s03_ses-01_task-spatialTS_run-1_events.tsv` in 
the BIDS directory.  If the first onsets look like this, they have not been adjusted for the trimming (only showing first few columns here, note I truncated some of the longer numbers)

| onset       | duration    | response_time | trial_id          |
|-------------|-------------|---------------|-------------------|
| 0.003       | n/a         | n/a           | fmri_trigger_wait |
| 10.006      | 1           | n/a           | fmri_buffer       |
| 11.00664767 | 0.73635233  | n/a           | test_fixation     |
| 11.745      | 0.15        | n/a           | blank_screen      |
| 11.898      | 1           | 0.911         | test_trial        |
| 13.89827933 | 0.10372067  | n/a           | test_fixation     |
| 14.003      | 0.15        | n/a           | blank_screen      |


**The TR comes from the fMRIPrep JSON sidecar, not from the config.** It is
cross-checked against the task YAML's `tr` and a disagreement raises. One run
in this dataset (`s19 ses-04 shapeMatching`) was acquired at TR 2.264 rather
than 1.49 — it will raise, and it should be excluded.

**The `cosine*` confounds ARE the high-pass filter.** They are fMRIPrep's
discrete cosine basis. Include them and you must not also pass `high_pass` or
`drift_model` to the GLM, or you filter twice. Exclude them and you need one of
those instead.

**Only the 8 base tasks are verified.** The dual-task YAMLs are not included.  If you need help setting those up, 
you should be able to learn the task `yaml` format and with Patrick's knowledge about how the task should be modeled
you can set up the design.  Jeanette is happy to help as well.

## 1. Setup

In [ ]:
from pathlib import Path
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import network_fmri.model_setup as dm

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

In [ ]:
# ---------------------------------------------------------------- PLACEHOLDERS
# Fill these in.
BIDS_DIR     = Path("/oak/stanford/groups/russpold/data/network_grant/discovery_BIDS_20250402") # BIDS root (events.tsv live here)
FMRIPREP_DIR = Path("/oak/stanford/groups/russpold/data/network_grant/discovery_BIDS_20250402"
                    "/derivatives/fmriprep_25.2.4")          # fMRIPrep derivatives root
TASK_YAML_DIR = Path("/oak/stanford/groups/russpold/users/jmumford/pfm/code/tasks")         # directory of <task>.yaml files

SUBJECT = "s03" 
SESSION = "ses-01"              
TASK    = "spatialTS"                 
RUN     = "run-1"                   

# The OAK derivatives are untrimmed with raw onsets. Leave False for them.
# WARNING:  DATA_IS_TRIMMED = True has NOT been tested.  When using for the first time, check
# trimmed data designs with the original untrimmed data designs to make sure they match.  
# If they don't, please report it to Jeanette or fix on your own.
DATA_IS_TRIMMED = False
# -----------------------------------------------------------------------------

TASK_YAML = TASK_YAML_DIR / f"{TASK}.yaml"

for p in [BIDS_DIR, FMRIPREP_DIR, TASK_YAML_DIR, TASK_YAML]:
    print(f"{'ok ' if p.exists() else 'MISSING'} {p}")

### Available tasks

Each task has a YAML defining its regressors and contrasts. Take a look at the
one you're using — the regressor definitions are readable and they are the main
thing determining what ends up in your design matrix.

In [ ]:
print("verified tasks:")
for p in sorted(TASK_YAML_DIR.glob("*.yaml")):
    print("  ", p.stem)

print(f"\n--- {TASK_YAML.name} ---")
print(TASK_YAML.read_text())

## 2. Building a design matrix

One call does everything: finds the files, reads the TR and slice-timing
reference from the sidecar, trims the warmup volumes, shifts the onsets, trims
the confounds, derives the junk-trial columns, convolves each regressor with
the SPM HRF, and assembles the matrix.

In [ ]:
res = dm.build_run_design_matrix(
    bids_dir=BIDS_DIR,
    fmriprep_dir=FMRIPREP_DIR,
    subject=SUBJECT,
    task=TASK,
    session=SESSION,
    run=RUN,
    task_yaml=TASK_YAML,
    confound_columns=dm.DEFAULT_CONFOUNDS,   # see section 3
    data_is_trimmed=DATA_IS_TRIMMED,
)

design = res["design_matrix"]
print(f"design matrix: {design.shape[0]} volumes x {design.shape[1]} columns")
design.head()

### What comes back

`build_run_design_matrix` returns a dict. Everything in it:

| key | what it is |
|---|---|
| `design_matrix` | the DataFrame you fit — regressors, then confounds, then `constant` |
| `contrasts` | `{name: formula}` from the task YAML, minus any referencing a dropped column |
| `dropped_columns` | regressors removed for having no variance (e.g. no omissions in this run) |
| `skipped_contrasts` | contrasts dropped because they referenced one of the above |
| `three_column` | per regressor, the `(onsets, durations, amplitudes)` actually used — useful for checking what got selected |
| `n_scans` | volumes **after** trimming; the number of design matrix rows |
| `tr` | from the sidecar |
| `start_time` | slice-timing reference in **seconds** (0.701 here), added to `frame_times` |
| `n_dummy` | warmup volumes actually removed (0 if `data_is_trimmed=True`) |
| `events` | the events DataFrame after the onset shift (no columns added) |
| `frame_times` | the sampling times used for convolution |
| `files` | the four paths that were read |
| `bold` | the trimmed timeseries — only when `return_bold=True` |

In [ ]:
print(f"n_scans           : {res['n_scans']}")
print(f"tr                : {res['tr']}")
print(f"start_time        : {res['start_time']}  (seconds, added to frame_times)")
print(f"n_dummy removed   : {res['n_dummy']}")
print(f"dropped columns   : {res['dropped_columns'] or 'none'}")
print(f"skipped contrasts : {res['skipped_contrasts'] or 'none'}")
print(f"\ncontrasts available:")
for name, formula in res["contrasts"].items():
    print(f"  {name:<28} {formula}")
print(f"\nfiles read:")
for k, v in res["files"].items():
    print(f"  {k:<11} {v.name}")

**`dropped_columns` is worth a glance every run.** A regressor with no events
in this run (no omissions, no false alarms) has no variance, so it is removed —
keeping it would make the design singular. Any contrast referring to it is
dropped too. That is normal and it reflects the subject's behaviour, but if a
regressor is empty in *every* run then its definition is probably wrong.

In [ ]:
# what each regressor actually selected, and with what durations
for name, (onsets, durations, amps) in res["three_column"].items():
    if len(onsets) == 0:
        print(f"{name:<34} NO EVENTS")
        continue
    nz = sum(1 for a in amps if a != 0)
    print(f"{name:<34} {len(onsets):>4} events ({nz} nonzero amp)  "
          f"durations {sorted(set(np.round(durations, 3)))[:4]}")

### Which events columns does this task need?

Because every name in the YAML is a real column, the loader can list exactly
what a config requires and check it against the events file **before** building
anything. A typo or a config written for a different events layout fails with
one clear message instead of partway through.

In [ ]:
config = dm.load_task_config(TASK_YAML)

print("columns this task config references:")
print(" ", dm.referenced_event_columns(config))

events = res["events"]
print("\ncolumns in the events file:")
print(" ", sorted(events.columns))

# this is what build_run_design_matrix calls for you
dm.check_event_columns(config, events, task_name=TASK_YAML.name)
print("\nall referenced columns are present")

In [ ]:
# how many events each regressor selects -- worth a glance
for name, cfg in config["regressors"].items():
    q = cfg["subset"]
    n = len(events.query(q)) if q else len(events)
    flag = "   <-- selects nothing" if n == 0 else ""
    print(f"{name:<24} {n:>5} events{flag}")
    print(f"{'':<24} amplitude={cfg['amplitude']!r}  duration={cfg['duration']!r}")

A regressor selecting **zero** events is worth checking. In a single run it
usually just means the subject didn't make that kind of error — an omission
regressor is legitimately empty for someone who responded to everything, and
it will be dropped as zero-variance. But a regressor that is empty in *every*
run means its subset is wrong, and nothing else will tell you.

## 3. Choosing confounds

`confound_columns` takes fMRIPrep confounds-TSV column names. Some named sets
are provided, and you can compose or extend them freely. An entry ending in
`*` is globbed against the TSV, which is needed for `cosine*` because fMRIPrep
emits a run-length-dependent number of DCT terms.

In [ ]:
print(f"dm.MOTION_6    ({len(dm.MOTION_6):>2}) {dm.MOTION_6}")
print(f"\ndm.MOTION_24   ({len(dm.MOTION_24):>2}) Friston-24: the 6 above plus")
print(f"                    derivative1, power2, derivative1_power2 of each")
print(f"\ndm.TISSUE      ({len(dm.TISSUE):>2}) {dm.TISSUE}")
print(f"dm.COSINE      ({len(dm.COSINE):>2}) {dm.COSINE}  <- this IS the high-pass filter")
print(f"\ndm.DEFAULT_CONFOUNDS = MOTION_24 + TISSUE + COSINE  ({len(dm.DEFAULT_CONFOUNDS)} specs)")

In [ ]:
# compose whatever you want
my_confounds = dm.MOTION_6 + dm.TISSUE + dm.COSINE
print(my_confounds)

# or add columns of your own -- a literal name that isn't in the TSV raises,
# so a typo fails immediately instead of silently shrinking your design
# my_confounds = dm.MOTION_24 + ["framewise_displacement", "std_dvars"] + dm.COSINE

### To see what's available in the TSV

Useful when you're deciding. fMRIPrep writes a lot of columns.

In [ ]:
conf_raw = pd.read_csv(res["files"]["confounds"], sep="\t", na_values=["n/a"])
print(f"{conf_raw.shape[0]} rows x {conf_raw.shape[1]} columns")
print(f"(rows = untrimmed length = n_scans {res['n_scans']} + {res['n_dummy']} warmup)\n")

groups = {}
for c in conf_raw.columns:
    key = ("motion" if c.startswith(("trans_", "rot_"))
           else "cosine" if c.startswith("cosine")
           else "aCompCor" if c.startswith("a_comp_cor")
           else "tCompCor" if c.startswith("t_comp_cor")
           else "outlier" if "outlier" in c
           else "other")
    groups.setdefault(key, []).append(c)
for k, v in groups.items():
    print(f"{k:<10} {len(v):>3}  {v[:4]}{' ...' if len(v) > 4 else ''}")

### Global signal

The `global_signal` column in the confounds TSV is a mean over the
**volumetric** brain mask, which is not the space you are modelling. Pass
`global_signal=True` and the module computes the mean over the actual
grayordinates of the trimmed dtseries instead.

This requires reading the timeseries (~120 MB per run), so it is off by
default. Optionally restrict it to cortex with `global_signal_structures`.

Whether to include global signal at all is a real methodological choice,
especially for functional connectivity — it removes global artifact but also
changes between-region correlation structure. It is opt-in for that reason.

In [ ]:
res_gs = dm.build_run_design_matrix(
    bids_dir=BIDS_DIR, fmriprep_dir=FMRIPREP_DIR,
    subject=SUBJECT, task=TASK, session=SESSION, run=RUN,
    task_yaml=TASK_YAML,
    confound_columns=dm.DEFAULT_CONFOUNDS,
    data_is_trimmed=DATA_IS_TRIMMED,
    global_signal=True,
    # global_signal_structures=["CIFTI_STRUCTURE_CORTEX_LEFT",
    #                           "CIFTI_STRUCTURE_CORTEX_RIGHT"],
)
gs = res_gs["design_matrix"]["global_signal"].to_numpy()
print(f"global_signal: mean {gs.mean():.1f}  sd {gs.std():.2f}  "
      f"range [{gs.min():.1f}, {gs.max():.1f}]")

fig, ax = plt.subplots(figsize=(13, 2.5))
ax.plot(gs, lw=0.9)
ax.set(xlabel="volume", ylabel="global signal",
       title="mean grayordinate timeseries (slow drift is expected)")
plt.tight_layout()

## 4. Plotting the design matrix

nilearn's `plot_design_matrix` scales each column to unit L2 norm for display.
That works well for regressors centred near zero, but columns whose mean dwarfs
their variance — `global_signal`, `csf`, `white_matter`, `constant` — come out
looking blank. They are not empty; the variation is just invisible next to the
offset.

Z-scoring each column for the plot fixes that.

In [ ]:
design = res_gs["design_matrix"]

summary = pd.DataFrame({
    "mean": design.mean(), "sd": design.std(),
    "min": design.min(), "max": design.max(),
})
summary["sd_over_mean"] = (summary.sd / summary["mean"].abs()).replace(np.inf, np.nan)
print("columns whose variance is tiny next to their mean (these render blank):")
print(summary[summary.sd_over_mean < 0.05].round(4).to_string())

In [ ]:
from nilearn.plotting import plot_design_matrix

# FOR DISPLAY ONLY -- see the warning below
design_for_display = design.apply(
    lambda c: (c - c.mean()) / c.std() if c.std() > 0 else c
)

plot_design_matrix(design_for_display, rescale=False)
plt.title("z-scored FOR DISPLAY ONLY -- do not fit this", fontsize=10)
plt.tick_params(labelsize=5)
plt.show()

> ### Do not fit `design_for_display`
>
> It exists only to make the picture readable. Fit `res["design_matrix"]`.
>
> Two reasons. The `constant` column has zero variance, so z-scoring leaves it
> unchanged while every other column is rescaled — the intercept would no
> longer be on a comparable footing. And your parameter estimates would be in
> units of each regressor's own standard deviation, which changes what a
> contrast value means and makes runs non-comparable, since the scaling factor
> differs run to run.
>
> Centring and scaling nuisance regressors does not change OLS residuals — the
> intercept absorbs the difference — so there is no benefit to fitting the
> rescaled version anyway.

## 5. Fitting the model and getting residuals

For residuals you want plain OLS. The efficient way with a (volumes ×
grayordinates) array is to solve once for all grayordinates at the same time
rather than looping: the design matrix is shared, so the pseudoinverse is
computed once and applied to the whole data matrix.

Set `return_bold=True` to get the trimmed timeseries back alongside the design.

In [ ]:
res_fit = dm.build_run_design_matrix(
    bids_dir=BIDS_DIR, fmriprep_dir=FMRIPREP_DIR,
    subject=SUBJECT, task=TASK, session=SESSION, run=RUN,
    task_yaml=TASK_YAML,
    confound_columns=dm.DEFAULT_CONFOUNDS,
    data_is_trimmed=DATA_IS_TRIMMED,
    global_signal=True,
    return_bold=True,          # <- gives you res["bold"]
)

Y = res_fit["bold"]                                  # (n_scans, 91282)
X = res_fit["design_matrix"].to_numpy(dtype=np.float64)
print(f"Y {Y.shape}  ({Y.nbytes / 1e6:.0f} MB, {Y.dtype})")
print(f"X {X.shape}")
assert Y.shape[0] == X.shape[0], "design and data must have the same number of volumes"

# Note, this OLS code should only be used to obtain residuals
If you're running an analysis an you need contrast estimates you must use prewhitening.  This will require adapting nilearn functions to run
a proper BOLD time series model on the data.  Jeanette is happy to help you figure this out.

In [ ]:
# OLS for every grayordinate at once.
#
# lstsq on the whole (T, V) right-hand side does one decomposition of X and
# applies it to all columns, which is far faster than a loop and more stable
# than forming (X'X)^-1 explicitly.
beta, *_ = np.linalg.lstsq(X, Y, rcond=None)         # (n_columns, 91282)
fitted = X @ beta
residuals = Y - fitted

print(f"beta      {beta.shape}")
print(f"residuals {residuals.shape}  ({residuals.nbytes / 1e6:.0f} MB)")

# sanity: residuals should be orthogonal to every design column
print(f"\nmax |X' * residuals| = {np.abs(X.T @ residuals).max():.2e}  (≈0 expected)")

# variance explained, per grayordinate
ss_tot = ((Y - Y.mean(axis=0)) ** 2).sum(axis=0)
ss_res = (residuals ** 2).sum(axis=0)
r2 = 1 - ss_res / np.maximum(ss_tot, 1e-12)
print(f"R^2 across grayordinates: median {np.median(r2):.3f}  "
      f"5th-95th pct [{np.percentile(r2, 5):.3f}, {np.percentile(r2, 95):.3f}]")

In [ ]:
# a look at one grayordinate
v = 10

fig, ax = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
ax[0].plot(Y[:, v], lw=0.9, label="data")
ax[0].plot(fitted[:, v], lw=0.9, label="fitted")
ax[0].legend(fontsize=8); ax[0].set_ylabel("signal")
ax[0].set_title(f"grayordinate {v}  (R² = {r2[v]:.3f})", fontsize=9)
ax[1].plot(residuals[:, v], lw=0.9, color="crimson")
ax[1].axhline(0, color="k", lw=0.5)
ax[1].set(xlabel="volume", ylabel="residual")
plt.tight_layout()

### Betas by name

`beta` rows are in design matrix column order, so you can index them by name.

In [ ]:
beta_df = pd.DataFrame(beta, index=res_fit["design_matrix"].columns)
print("mean beta across grayordinates, task regressors only:")
task_names = [c for c in beta_df.index
              if c in dm.load_task_config(TASK_YAML)["regressors"]]
print(beta_df.loc[task_names].mean(axis=1).round(4).to_string())

### Optional: band-pass filtering the residuals

For functional connectivity, residuals are often band-pass filtered. The
previous pipeline used 0.01–0.1 Hz via `nilearn.signal.clean`.

Note this interacts with your confound choice: if the `cosine*` columns are
already in the design, the high-pass has been applied by regression, and adding
`high_pass` here filters twice. Decide which mechanism you want, not both.

In [ ]:
from nilearn.signal import butterworth

# This assumes a .1 lowpass filter, change if needed
residuals_bp = butterworth(
    residuals,
    sampling_rate=1 / res_fit["tr"],   # note: Hz, not TR
    low_pass=0.1,
    high_pass=None,
    copy=True,                          
)
print(f"filtered residuals {residuals_bp.shape}")
print(f"sd before {residuals.std():.4f}  after {residuals_bp.std():.4f}")

### Saving

Left to you. A couple of pointers:

- The trimmed residuals are shorter than the original dtseries, so the header's
  series axis no longer describes them. To write a valid CIFTI you need a new
  `SeriesAxis` with `size=n_scans` and, if you want the timing honest,
  `start` advanced by `n_dummy * tr`. `res["files"]["dtseries"]` is the file to
  copy the brain-model axis from.
- The design matrix is a plain DataFrame: `res["design_matrix"].to_csv(...)`.
  Worth saving alongside any results, since it records exactly what was
  regressed out.

## Looping over runs

The pattern for a whole study. Catch exceptions per run so one bad run doesn't
end the loop — and do look at what failed rather than ignoring it.

In [ ]:
# Replace with however you enumerate your runs.
# FILL THIS IN TO RUN IT.  
# I'd write a script that generates this runs list based on the BIDS directory 
# and the task list that you're using, so you don't have to type it by hand.
runs = [
    # dict(subject="s03", session="ses-01", task="flanker", run="run-1"),
]

results, failures = [], []
for r in runs:
    try:
        out = dm.build_run_design_matrix(
            bids_dir=BIDS_DIR, fmriprep_dir=FMRIPREP_DIR,
            task_yaml=TASK_YAML_DIR / f"{r['task']}.yaml",
            confound_columns=dm.DEFAULT_CONFOUNDS,
            data_is_trimmed=DATA_IS_TRIMMED,
            load_data=False,          # header only: fast, no timeseries read
            **r,
        )
        results.append({**r, "n_scans": out["n_scans"],
                        "n_columns": out["design_matrix"].shape[1],
                        "dropped": ", ".join(out["dropped_columns"]),
                        "skipped": ", ".join(out["skipped_contrasts"])})
    except Exception as e:
        failures.append({**r, "error": f"{type(e).__name__}: {e}"})

if results:
    display(pd.DataFrame(results))
if failures:
    print(f"\n{len(failures)} failure(s):")
    display(pd.DataFrame(failures))

`load_data=False` reads only the dtseries header. The design matrix takes
nothing from the timeseries but its length, so this is the fast path when you
are building or checking many designs. You need the data (so `load_data` left
at its default, or `return_bold=True`, or `global_signal=True`) only when you
are actually fitting.